# M1 — arXiv crawler (Colab + runtime tracking)

Quy trình: GitHub branch → Drive → Fast Crawl → Checkpoint → Recovery → Report.

**Mới:** Cell 4 và Cell 6 tự đo thời gian thực tế rồi ghi `runtime_history.csv` trong thư mục Drive. Cell 7 chỉ đọc và tổng hợp, không cộng trùng. **Cần push các file Python trong bản vá lên nhánh `M1-update` trước khi chạy notebook.**

Các lần crawl trước khi cập nhật không tự xuất hiện trong lịch sử thời gian.


## 1 — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuanadtedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-update"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/m1_crawling/requirements-crawler.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_arxiv_crawler.py", "tests/test_runtime_tracking.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)


## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3 — Chọn khoảng ID và thư mục lưu

In [ ]:
RANGE_ID = 5             # 1..5, mỗi thành viên chọn range được giao
BATCH_SIZE = 50          # giữ nguyên khi chạy lại / Recovery
DATA_DIR = Path(f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Range:", RANGE_ID, "| Batch size:", BATCH_SIZE, "| Drive:", DATA_DIR)


## 4 — Fast Crawl: không retry, tự lưu checkpoint

`batches=1` là một batch 50 ID. Gọi lại hàm để chạy các batch tiếp theo. Nếu arXiv báo giới hạn truy cập, dừng và chờ thay vì tăng tốc/đa luồng.

In [ ]:
import time
from src.m1_crawling.runtime_tracking import (
    checkpoint_state, record_runtime, utc_now,
)


def _run_crawler(extra_args, *, mode, batches_requested):
    cmd = [sys.executable, "-m", "src.m1_crawling.run_arxiv_only",
           "--range", str(RANGE_ID),
           "--batch-size", str(BATCH_SIZE),
           "--backup-dir", str(DATA_DIR),
           "--delay", "3.1", *extra_args]
    before = checkpoint_state(DATA_DIR)
    started_at = utc_now()
    start = time.perf_counter()
    exit_code = -1
    try:
        result = subprocess.run(cmd, cwd=PROJECT)
        exit_code = result.returncode
    finally:
        elapsed = time.perf_counter() - start
        try:
            event = record_runtime(
                DATA_DIR, mode=mode, range_id=RANGE_ID,
                batch_size=BATCH_SIZE, batches_requested=batches_requested,
                elapsed_seconds=elapsed, exit_code=exit_code,
                started_at_utc=started_at, before=before,
                after=checkpoint_state(DATA_DIR),
            )
            print(f"[RUNTIME] {mode}: {elapsed:.1f}s "
                  f"({elapsed / 60:.2f} phút), trạng thái={event['result']}")
        except OSError as exc:
            print(f"[WARNING] Không lưu được runtime lên Drive: {exc}")
    # Code 2 = có source lỗi, checkpoint vẫn được lưu.
    if exit_code not in (0, 2):
        raise RuntimeError(f"Crawler failed with exit code {exit_code}")
    return exit_code


def fast_crawl(batches=1):
    return _run_crawler(["--next", str(batches), "--retries", "0"],
                        mode="fast", batches_requested=batches)

# Thử 1 batch; sau đó có thể gọi fast_crawl(batches=5) để tiếp tục.
fast_crawl(batches=1)


## 5 — Kiểm tra các checkpoint đã lưu

Việc lưu source + metadata xảy ra **trong lúc crawl**, không phải đợi đến cell này.

In [ ]:
import json

def show_checkpoints():
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Đã lưu {len(manifests)} checkpoint")
    for marker in manifests:
        info = json.loads(marker.read_text(encoding="utf-8"))
        archive = marker.with_suffix(".tar.gz")
        if not archive.exists():
            print("MISSING ARCHIVE:", marker.name)
            continue
        print(f"{marker.stem}: ok={info.get('ok')}, error={info.get('failed')}, "
              f"no_tex={info.get('no_tex')}, size={archive.stat().st_size/1024**2:.2f} MiB")

show_checkpoints()


## 6 — Recovery: chỉ tải lại phiên bản lỗi, KHÔNG tự sao lưu bản cũ

- Mặc định `backup_before=False`: tránh sao chép toàn bộ `.tar.gz` vào `_before_recovery` và làm đầy Google Drive.
- Checkpoint hiện tại vẫn được cập nhật sau Recovery; `no_tex` không retry.
- Nếu thực sự cần giữ bản cũ để đối chiếu, đặt `backup_before=True` **chỉ cho một lần** và đảm bảo Drive còn đủ dung lượng.
- `batches=1` nghĩa là retry tối đa **1 batch có lỗi**, không phải 1 bài báo.


In [ ]:
import json
import shutil

def recovery_crawl(batches=1, retries=0, backup_before=False):
    if backup_before:
        snapshot = DATA_DIR / "_before_recovery"
        snapshot.mkdir(parents=True, exist_ok=True)
        copied = 0
        for marker in sorted(DATA_DIR.glob("batch_*.json")):
            info = json.loads(marker.read_text(encoding="utf-8"))
            if int(info.get("failed", 0)) <= 0:
                continue
            archive = marker.with_suffix(".tar.gz")
            if not archive.exists():
                raise FileNotFoundError(f"Missing archive: {archive}")
            for source in (marker, archive):
                destination = snapshot / source.name
                if not destination.exists():
                    shutil.copy2(source, destination)
                    copied += 1
        print(f"Saved {copied} original checkpoint files in {snapshot}")
    return _run_crawler(["--next", str(batches), "--retry-failed",
                         "--retries", str(retries)],
                        mode="recovery", batches_requested=batches)

# Chỉ chạy khi cần Recovery; KHÔNG tạo bản sao mặc định.
recovery_crawl(batches=1, retries=0)


## 7 — Export report: lỗi + thời gian tích lũy

- `runtime_history.csv`: từng lần Fast/Recovery và số giây thực tế.
- `summary.json`: `fast_crawl_seconds`, `recovery_seconds`, `total_runtime_seconds`, `recovery_attempts`, `average_seconds_per_id`.
- `issues_explained_vi.csv`: giải thích lỗi theo phiên bản.
- Export nhiều lần không ghi thêm runtime.


In [ ]:
from src.m1_crawling.export_reports import export_reports


def export_report(verify_sha256=False):
    summary = export_reports(DATA_DIR, verify_sha256=verify_sha256)
    print("===== REPORT =====")
    for key in ("batch_count", "total_records", "metadata_rows", "ok", "error",
                "no_tex", "not_found", "failed_version_rows"):
        print(f"{key}: {summary[key]}")
    print("Loai van de:", summary.get("issue_counts", {}))
    print("===== RUNTIME (từ khi cài bản vá) =====")
    for key in ("fast_crawl_seconds", "recovery_seconds", "total_runtime_seconds",
                "fast_crawl_runs", "recovery_attempts", "average_seconds_per_id"):
        print(f"{key}: {summary.get(key)}")
    print("Lịch sử:", DATA_DIR / "reports" / "runtime_history.csv")
    print("Checkpoint: %.2f MiB" % (summary.get("checkpoint_archive_bytes", 0) / 1024**2))
    print("Ban sao Recovery: %.2f MiB" % (summary.get("recovery_snapshot_bytes", 0) / 1024**2))
    print("Saved at:", DATA_DIR / "reports")
    for path in sorted((DATA_DIR / "reports").iterdir()):
        print(" -", path.name)
    return summary

summary = export_report()
print("\n===== GIAI THICH REPORT =====")
guide = DATA_DIR / "reports" / "README_BAO_CAO.txt"
if guide.exists():
    print(guide.read_text(encoding="utf-8"))
else:
    print("Chưa có README_BAO_CAO.txt; các báo cáo CSV vẫn được xuất.")


## 8 — Kiểm tra dung lượng và dọn bản sao Recovery CŨ (tùy chọn)

`_before_recovery` chứa bản sao trước Recovery; nó không cần cho việc tiếp tục crawl hay xuất report. **Đừng xóa** nếu bạn còn cần khôi phục phiên bản cũ. Hàm dưới đây mặc định chỉ xem trước, không xóa gì.


In [ ]:
from pathlib import Path
import json
import shutil

def check_recovery_snapshot(delete=False):
    snapshot = DATA_DIR / "_before_recovery"
    if not snapshot.exists():
        print("Không có _before_recovery, không cần dọn.")
        return
    # Verify active archives exist and have the sizes recorded in manifests.
    for marker in DATA_DIR.glob("batch_*.json"):
        manifest = json.loads(marker.read_text(encoding="utf-8"))
        archive = marker.with_suffix(".tar.gz")
        if not archive.is_file() or archive.stat().st_size != manifest.get("archive_bytes"):
            raise RuntimeError(f"Checkpoint hiện tại chưa hợp lệ: {marker.name}; không xóa snapshot")
    size = sum(p.stat().st_size for p in snapshot.rglob("*") if p.is_file())
    print(f"Bản sao cũ: {snapshot} ({size / 1024**2:.2f} MiB)")
    if delete:
        shutil.rmtree(snapshot)
        print("Đã xóa bản sao cũ; checkpoint chính vẫn được giữ nguyên.")
    else:
        print("Chưa xóa gì. Nếu không cần khôi phục bản cũ, chạy check_recovery_snapshot(delete=True).")

check_recovery_snapshot(delete=False)


## Sau khi chạy

- Gửi cho người xử lý dữ liệu: `reports/papers.csv`, `reports/crawl_records.csv`, **archive `batch_*.tar.gz`** và đường dẫn Drive.
- Đọc `reports/issues_explained_vi.csv` để hiểu lỗi, tránh retry vô ích với `no_tex`.
- Mặc định không sao lưu bản cũ mỗi Recovery để tiết kiệm Drive.
- Muốn crawl toàn bộ range theo checkpoint có thể gọi `fast_crawl(batches=0)` (`--next 0` = tất cả batch còn thiếu); Colab có thể ngắt phiên, lần sau tiếp tục.
